In [ ]:
pip install h5py

In [1]:
import gc
import h5py
import math
import matplotlib.pyplot as plt
import os
import psutil
import sys
import torch
import torch.nn.init as init
import uuid


def nice_print(**kwargs):
    for k, v in kwargs.items():
        if isinstance(v, torch.Tensor):
            print(f'Tensor "{k}" has shape {v.shape}')
        else:
            print(f'Variable "{k}" has value `{v}`')


def print_shape(o):
    dims = []
    while True:
        try:
            dims.append(len(o))
            o = next(iter(o))
        except:
            print(dims)
            return


def mem_report():
    for obj in gc.get_objects():
        if torch.is_tensor(obj):
            print(type(obj), obj.size())


def cpu_stats():
    print(sys.version)
    print(psutil.cpu_percent())
    print(psutil.virtual_memory())  # physical memory usage
    pid = os.getpid()
    py = psutil.Process(pid)
    memoryUse = py.memory_info()[0] / 2.0 ** 30  # memory use in GB...I think
    print("memory GB:", memoryUse)


def grouper(iterable, n):
    """Collect data into fixed-length chunks or blocks
       E.g., grouper('ABCDEFG', 3) --> ABC DEF
    """
    args = [iter(iterable)] * n
    return zip(*args)


def window(seq, size=2, stride=1):
    """Returns a sliding window (of width n) over data from the iterable
       E.g., s -> (s0,s1,...s[n-1]), (s1,s2,...,sn), ...  
    """
    it = iter(seq)
    result = []
    for elem in it:
        result.append(elem)
        if len(result) == size:
            yield result
            result = result[stride:]


def draw(imgs):
    size = len(imgs)
    fig, axs = plt.subplots(2, size, figsize=(5, 5), constrained_layout=True)
    for img, ax1, ax2 in zip(imgs, axs[0], axs[1]):
        ax1.imshow(img[0])
        ax2.imshow(img[1])
    plt.show()


def weights_init(init_type="gaussian"):
    def init_fun(m):
        classname = m.__class__.__name__
        if (classname.find("Conv") == 0 or classname.find("Linear") == 0) and hasattr(
            m, "weight"
        ):
            # print m.__class__.__name__
            if init_type == "gaussian":
                init.normal_(m.weight.data, 0.0, 0.02)
            elif init_type == "xavier":
                init.xavier_normal_(m.weight.data, gain=math.sqrt(2))
            elif init_type == "kaiming":
                init.kaiming_normal_(m.weight.data, a=0, mode="fan_in")
            elif init_type == "orthogonal":
                init.orthogonal_(m.weight.data, gain=math.sqrt(2))
            elif init_type == "default":
                pass
            else:
                assert 0, "Unsupported initialization: {}".format(init_type)
            if hasattr(m, "bias") and m.bias is not None:
                init.constant_(m.bias.data, 0.0)

    return init_fun


def h5_virtual_file(filenames, name="data"):
    """
    Assembles a virtual h5 file from multiples
    """
    vsources = []
    total_t = 0
    for path in filenames:
        data = h5py.File(path, "r").get(name)
        t, *features_shape = data.shape
        total_t += t
        vsources.append(h5py.VirtualSource(path, name, shape=(t, *features_shape)))

    # Assemble virtual dataset
    layout = h5py.VirtualLayout(shape=(total_t, *features_shape), dtype=data.dtype)
    cursor = 0
    for vsource in vsources:
        # we generate slices like layour[0:10,:,:,:]
        indices = (slice(cursor, cursor + vsource.shape[0]),) + (slice(None),) * (
            len(vsource.shape) - 1
        )
        layout[indices] = vsource
        cursor += vsource.shape[0]
    # Add virtual dataset to output file
    f = h5py.File(f"{uuid.uuid4()}.h5", "w", libver="latest")
    f.create_virtual_dataset(name, layout)
    return f

C:\Users\uceti\anaconda3\envs\CUDA-pytorch\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal, Categorical
from torch.utils.data.sampler import BatchSampler, SubsetRandomSampler
from torch.nn.modules.utils import _pair
from torch.autograd import Variable
import sklearn
from sklearn.model_selection import train_test_split
from sklearn import metrics
from functools import reduce
#from utils import nice_print, mem_report, cpu_stats
import copy
import operator
import numpy as np
from einops import rearrange, repeat
from torch import nn, einsum



class SAE3DLSTM(nn.Module):
    '''
    tau:timstamp-1
    input:(n,b,c,t,h,w) n:fragments;t:frames in the fragments;
    output:(b,c,t,h,w)'''

    def __init__(self, input_shape, hidden_size, num_layers, kernel_size,reduce_channel):
        super().__init__()
        self.agg = nn.Parameter(torch.randn(1, input_shape[0], input_shape[1], input_shape[2], input_shape[3])) #c,t,h,w
        # self._tau = n
        self._cells = []

        input_shape = list(input_shape)
        for i in range(num_layers):
            cell = SASAE3DLSTMCell(input_shape, hidden_size, kernel_size,self.agg,reduce_channel)
            # NOTE hidden state becomes input to the next cell
            input_shape[0] = hidden_size
            self._cells.append(cell)
            # Hook to register submodule
            setattr(self, "cell{}".format(i), cell)

    def forward(self, input):
        # NOTE (seq_len, batch, input_shape)
        batch_size = input.size(1)
        c_history_states = []
        h_states = []
        outputs = []
        seq_len=input.shape[0]
        # self.position_emb = _get_sinusoid_encoding_table(seq_len-1,)

        for step, x in enumerate(input):
            for cell_idx, cell in enumerate(self._cells):
                if step == 0: #
                    c_history, m, h = self._cells[cell_idx].init_hidden(
                        batch_size,seq_len-1, input.device
                    )
                    c_history_states.append(c_history)
                    h_states.append(h)

                # NOTE c_history and h are coming from the previous time stamp, but we iterate over cells
                c_history, m, h = cell(
                    x, c_history_states[cell_idx], m, h_states[cell_idx]
                )
                c_history_states[cell_idx] = c_history
                h_states[cell_idx] = h
                # NOTE hidden state of previous LSTM is passed as input to the next one
                x = h
            outputs.append(h)

        return outputs[-1]


class SASAE3DLSTMCell(nn.Module):
    def __init__(self, input_shape, hidden_size, kernel_size,agg,reduce_channel):
        super().__init__()
        #input shape [c,t,h,w]
        in_channels = input_shape[0]
        self._input_shape = input_shape
        self._hidden_size = hidden_size

        # memory gates: input, cell(input modulation), forget
        self.weight_xi = ConvDeconv3d(in_channels, hidden_size, kernel_size)
        self.weight_hi = ConvDeconv3d(hidden_size, hidden_size, kernel_size, bias=False)

        self.weight_xg = copy.deepcopy(self.weight_xi)
        self.weight_hg = copy.deepcopy(self.weight_hi)


        '''my defination'''
        self.agg=agg
        self.reduce_channel=reduce_channel
        self.conv=nn.Conv3d(input_shape[0],self.reduce_channel,kernel_size=(input_shape[1],1,1))
        dim=self.reduce_channel*input_shape[2]*input_shape[3] #c,h,w
        self.to_qk = nn.Linear(dim, dim * 2, bias = False)
        self.scale=dim** -0.5

        memory_shape = list(input_shape)
        memory_shape[0] = hidden_size

        self.layer_norm = nn.LayerNorm(memory_shape)

        # for spatiotemporal memory
        self.weight_xi_prime = copy.deepcopy(self.weight_xi)
        self.weight_mi_prime = copy.deepcopy(self.weight_hi)

        self.weight_xg_prime = copy.deepcopy(self.weight_xi)
        self.weight_mg_prime = copy.deepcopy(self.weight_hi)

        self.weight_xf_prime = copy.deepcopy(self.weight_xi)
        self.weight_mf_prime = copy.deepcopy(self.weight_hi)

        self.weight_xo = copy.deepcopy(self.weight_xi)
        self.weight_ho = copy.deepcopy(self.weight_hi)
        self.weight_co = copy.deepcopy(self.weight_hi)
        self.weight_mo = copy.deepcopy(self.weight_hi)

        self.weight_111 = nn.Conv3d(hidden_size + hidden_size, hidden_size, 1)

    def attn(self,q, k, v):
        sim = einsum('b i d, b j d -> b i j', q, k)
        attn = sim.softmax(dim=-1)
        attn=attn.squeeze(1)
        out = einsum('b j, j b c t h w -> b c t h w', attn, v)
        return out

    def self_attention_qk(self, agg,x,h,c_history):
        b,_,_,_,_=x.shape
        #value
        agg_batch=agg.repeat(b,1,1,1,1)
        agg_value=agg_batch.unsqueeze(0)
        x_value=x.unsqueeze(0)
        h_value = h.unsqueeze(0)
        value=torch.cat((agg_value,x_value,h_value,c_history),0)# n, b ,c ,t ,h ,w

        #reduce dimension
        x_reduce=self.conv(x)
        x_reduce = torch.squeeze(x_reduce,2)
        h_reduce=self.conv(h)
        h_reduce = torch.squeeze(h_reduce,2)
        agg_reduce=self.conv(agg_batch)
        agg_reduce = torch.squeeze(agg_reduce,2)
        c_history=rearrange(c_history,'n b c t h w -> (n b) c t h w')
        c_reduce=self.conv(c_history)
        c_reduce = torch.squeeze(c_reduce,2)
        c_reduce=rearrange(c_reduce,'(n b) c h w -> b n (c h w)',b=b)

        #flatten
        x_reduce = torch.flatten(x_reduce,1)#b,chw
        h_reduce = torch.flatten(h_reduce, 1)  # b,chw
        agg_reduce = torch.flatten(agg_reduce, 1)  # b,chw
        x_reduce = x_reduce.unsqueeze(1)
        h_reduce = h_reduce.unsqueeze(1)
        agg_reduce = agg_reduce.unsqueeze(1)
        #cat
        sequence=torch.cat((agg_reduce,x_reduce,h_reduce,c_reduce),1)
        #q,k
        q, k= self.to_qk(sequence).chunk(2, dim=-1)
        q = q * self.scale
        (agg_q, q_), (agg_k, k_) = map(lambda t: (t[:, :1], t[:, 1:]), (q, k))
        s= self.attn(agg_q, k, value)
        s = torch.squeeze(s)
        return s

    def forward(self, x, c_history, m, h):
        # Normalized shape for LayerNorm is CxT×H×W
        # print(x.shape,h.shape,'x h')
        normalized_shape = list(h.shape[-3:])

        def LR(input):
            return F.layer_norm(input, normalized_shape)

        i = torch.sigmoid(LR(self.weight_xi(x) + self.weight_hi(h)))
        g = torch.tanh(LR(self.weight_xg(x) + self.weight_hg(h)))

        s =self.self_attention_qk(self.agg,x, h,c_history)
        # nice_print(**locals())
        # mem_report()
        # cpu_stats()
        # print(s.shape,1)
        c = i * g + self.layer_norm(c_history[-1] + s) #distill useful information

        i_prime = torch.sigmoid(LR(self.weight_xi_prime(x) + self.weight_mi_prime(m)))
        g_prime = torch.tanh(LR(self.weight_xg_prime(x) + self.weight_mg_prime(m)))
        f_prime = torch.sigmoid(LR(self.weight_xf_prime(x) + self.weight_mf_prime(m)))

        m = i_prime * g_prime + f_prime * m
        o = torch.sigmoid(
            LR(
                self.weight_xo(x)
                + self.weight_ho(h)
                + self.weight_co(c)
                + self.weight_mo(m)
            )
        )
        h = o * torch.tanh(self.weight_111(torch.cat([c, m], dim=1)))

        # TODO is it correct FIFO?
        c_history = torch.cat([c_history[1:], c[None, :]], dim=0)
        # nice_print(**locals())
        return (c_history, m, h)

    def init_hidden(self, batch_size, n, device=None):
        memory_shape = list(self._input_shape)
        memory_shape[0] = self._hidden_size
        c_history = torch.zeros(n, batch_size, *memory_shape, device=device)
        m = torch.zeros(batch_size, *memory_shape, device=device)
        h = torch.zeros(batch_size, *memory_shape, device=device)

        return (c_history, m, h)


class ConvDeconv3d(nn.Module):
    def __init__(self, in_channels, out_channels, *vargs, **kwargs):
        super().__init__()

        self.conv3d = nn.Conv3d(in_channels, out_channels, *vargs, **kwargs)
        # self.conv_transpose3d = nn.ConvTranspose3d(out_channels, out_channels, *vargs, **kwargs)

    def forward(self, input):
        # print(self.conv3d(input).shape, input.shape)
        # return self.conv_transpose3d(self.conv3d(input))
        return F.interpolate(self.conv3d(input), size=input.shape[-3:], mode="nearest")


class Model(nn.Module):
    def __init__(self,window=3,frame=6,reduce_channel=2,width=12,height=16,hidden_units=32,out_channel=2):
        super().__init__()
        self.window=window
        self.width=width
        self.height=height
        self.out_channel=out_channel
        self.sae3d = nn.Sequential(
            SAE3DLSTM(input_shape=(hidden_units, frame, width, height), hidden_size=hidden_units, num_layers=2, kernel_size=(3, 3, 3),reduce_channel=reduce_channel),
            nn.ReLU(),
        )

        self.conv3d_encoder = nn.Sequential(
            nn.Conv3d(2, hidden_units, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(),
            nn.Conv3d(hidden_units, hidden_units, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU()
        )

        self.conv3d_decoder = nn.Sequential(
            nn.Conv3d(hidden_units, hidden_units, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(),
            nn.Conv3d(hidden_units, 2, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
        )

        self.vector = nn.Sequential(
            nn.Linear(56, 10),
            nn.ReLU(),
            nn.Linear(10, self.out_channel * width * height),
        )

        self.map= nn.Sequential(
            nn.Conv2d(2*frame,2,1)
        )

    def forward(self, video, x2,y=None): #x2:external factors
        # encoder
        video = rearrange(video, 'b (n t) c h w -> (n b) c t h w ', n=self.window)
        video= self.conv3d_encoder(video)
        video = rearrange(video, '(n b) c t h w -> n b c t h w ', n=self.window)
        video = self.sae3d(video)
        video=self.conv3d_decoder(video)
        video = rearrange(video, 'b c t h w -> b (c t) h w')
        video = self.map(video)
        # external factors
        x2 = self.vector(x2)
        x2=torch.reshape(x2,(-1,self.out_channel,self.width,self.height))
        out = video+x2
        if y is not None:
            loss=F.mse_loss(out, y)
            return loss
        else:
            return out

if __name__ == '__main__':
    video = np.ones((2, 12, 2,12, 16))  # b (n t) c h w。t:frames in the fragment; n:fragments,i.e.,closeness,period,trend; c:channel; h:height; h:width; b: batch
    video = torch.tensor(video, dtype=torch.float)
    x2 = np.ones((2, 56))  #x2:external factors
    x2 = torch.tensor(x2, dtype=torch.float)
    model = Model(3,4)
    y = model(video, x2)
    print(y.shape)


torch.Size([2, 2, 12, 16])


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from datetime import datetime
from tqdm import tqdm

# Custom Dataset Class
class VideoDataset(Dataset):
    def __init__(self, video_data, external_data, targets=None):
        """
        Args:
            video_data: numpy array of shape (num_samples, n*t, c, h, w)
            external_data: numpy array of shape (num_samples, external_features)
            targets: numpy array of shape (num_samples, out_channel, h, w)
        """
        self.video_data = torch.FloatTensor(video_data)
        self.external_data = torch.FloatTensor(external_data)
        self.targets = torch.FloatTensor(targets) if targets is not None else None
        self.has_targets = targets is not None
        
    def __len__(self):
        return len(self.video_data)
    
    def __getitem__(self, idx):
        if self.has_targets:
            return self.video_data[idx], self.external_data[idx], self.targets[idx]
        return self.video_data[idx], self.external_data[idx]

# Training Function
def train_model(model, train_loader, val_loader, config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    
    for epoch in range(config['epochs']):
        # Training phase
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']} [Train]")
        
        for videos, externals, targets in progress_bar:
            videos = videos.to(device)
            externals = externals.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            loss = model(videos, externals, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
        
        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for videos, externals, targets in val_loader:
                videos = videos.to(device)
                externals = externals.to(device)
                targets = targets.to(device)
                
                loss = model(videos, externals, targets)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(config['save_dir'], 'best_model.pth'))
            print(f"Saved new best model with val loss: {val_loss:.4f}")
    
    # Save final model and training stats
    torch.save(model.state_dict(), os.path.join(config['save_dir'], 'final_model.pth'))
    np.save(os.path.join(config['save_dir'], 'train_losses.npy'), np.array(train_losses))
    np.save(os.path.join(config['save_dir'], 'val_losses.npy'), np.array(val_losses))
    
    return model, train_losses, val_losses

# Configuration
config = {
    'batch_size': 32,
    'epochs': 100,
    'lr': 1e-3,
    'window': 3,         
    'frame': 6,          
    'reduce_channel': 2, 
    'width': 12,         
    'height': 16,        
    'hidden_units': 32,  
    'out_channel': 2,    
    'save_dir': 'saved_models',
    'val_split': 0.2,
    'random_seed': 42
}

# Prepare directories
os.makedirs(config['save_dir'], exist_ok=True)

# Example usage with dummy data
if __name__ == '__main__':
    # Generate dummy data - replace with your actual data loading
    num_samples = 1000
    video_data = np.random.rand(num_samples, config['window']*config['frame'], 2, config['width'], config['height'])
    external_data = np.random.rand(num_samples, 56)  # 56 matches your model's expectation
    targets = np.random.rand(num_samples, config['out_channel'], config['width'], config['height'])
    
    # Split into train and validation
    train_idx, val_idx = train_test_split(np.arange(num_samples), test_size=config['val_split'], random_state=config['random_seed'])
    
    train_dataset = VideoDataset(video_data[train_idx], external_data[train_idx], targets[train_idx])
    val_dataset = VideoDataset(video_data[val_idx], external_data[val_idx], targets[val_idx])
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
    
    # Initialize model
    model = Model(
        window=config['window'],
        frame=config['frame'],
        reduce_channel=config['reduce_channel'],
        width=config['width'],
        height=config['height'],
        hidden_units=config['hidden_units'],
        out_channel=config['out_channel']
    )
    
    # Train the model
    trained_model, train_losses, val_losses = train_model(model, train_loader, val_loader, config)
    
    print("Training completed!")

Epoch 1/100 [Train]: 100%|██████████| 25/25 [00:49<00:00,  1.97s/it, loss=0.1]  


Epoch 1: Train Loss: 0.1465, Val Loss: 0.0999
Saved new best model with val loss: 0.0999


Epoch 2/100 [Train]: 100%|██████████| 25/25 [00:48<00:00,  1.93s/it, loss=0.0848]


Epoch 2: Train Loss: 0.0894, Val Loss: 0.0852
Saved new best model with val loss: 0.0852


Epoch 3/100 [Train]: 100%|██████████| 25/25 [00:44<00:00,  1.77s/it, loss=0.0837]


Epoch 3: Train Loss: 0.0841, Val Loss: 0.0844
Saved new best model with val loss: 0.0844


Epoch 4/100 [Train]: 100%|██████████| 25/25 [00:44<00:00,  1.77s/it, loss=0.0839]


Epoch 4: Train Loss: 0.0837, Val Loss: 0.0843
Saved new best model with val loss: 0.0843


Epoch 5/100 [Train]:  52%|█████▏    | 13/25 [00:25<00:23,  1.97s/it, loss=0.0834]


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchsummary import summary

# Simulated video tensor: (batch_size, fragments * frames, channels, height, width)
video = np.ones((2, 2, 2, 12, 16))  # b (n t) c h w
video = torch.tensor(video, dtype=torch.float32)

# External factors tensor: (batch_size, features)
x2 = np.ones((2, 56))  # Example external features
x2 = torch.tensor(x2, dtype=torch.float32)

# Define a dummy model
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv = nn.Conv3d(in_channels=2, out_channels=4, kernel_size=3, padding=1)
        self.fc = nn.Linear(56, 10)  # Assume x2 has 56 features, output mapped to 10

    def forward(self, video, x2):
        video_out = self.conv(video)  # Process video input
        x2_out = self.fc(x2)  # Process external factors
        return video_out, x2_out  # Output both processed tensors

# Initialize model
model = Model()

# Instead of using summary() directly, manually print input-output shapes
video_out, x2_out = model(video, x2)
print("Video Output Shape:", video_out.shape)
print("External Factors Output Shape:", x2_out.shape)
